In [13]:
from dotenv import load_dotenv
load_dotenv()

import pdfplumber
from openai import OpenAI


In [14]:
PDF_PATH = "data/company_overview.pdf"
SMALL_TABLE_ROW_THRESHOLD = 10  # tables with FEWER rows than this get summarized as 1 chunk

In [15]:
def extract_tables_and_text(pdf_path):
    """Extract raw tables and full page text from every page of the PDF."""
    all_tables = []
    full_text_pages = []
 
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            tables = page.extract_tables()
            for table in tables:
                if table and len(table) > 0:
                    all_tables.append({
                        "page": page_num,
                        "raw": table,          # list of rows, each row = list of cells
                        "header": table[0],
                        "data_rows": table[1:],  # excludes header row
                    })
 
            text = page.extract_text() or ""
            full_text_pages.append({"page": page_num, "text": text})
 
    return all_tables, full_text_pages

In [16]:
all_tables, full_text_pages = extract_tables_and_text(PDF_PATH)

In [17]:
all_tables

[{'page': 1,
  'raw': [['Plan', 'Price', 'Users'],
   ['Free', '$0', '1'],
   ['Pro', '$20', '5'],
   ['Team', '$50', '20'],
   ['Enterprise', '$200', '100']],
  'header': ['Plan', 'Price', 'Users'],
  'data_rows': [['Free', '$0', '1'],
   ['Pro', '$20', '5'],
   ['Team', '$50', '20'],
   ['Enterprise', '$200', '100']]}]

In [18]:
all_tables[0]

{'page': 1,
 'raw': [['Plan', 'Price', 'Users'],
  ['Free', '$0', '1'],
  ['Pro', '$20', '5'],
  ['Team', '$50', '20'],
  ['Enterprise', '$200', '100']],
 'header': ['Plan', 'Price', 'Users'],
 'data_rows': [['Free', '$0', '1'],
  ['Pro', '$20', '5'],
  ['Team', '$50', '20'],
  ['Enterprise', '$200', '100']]}

In [19]:
def table_to_markdown(table):
    """
    Convert a pdfplumber-extracted table dictionary into a Markdown table.

    Expected format:
    {
        "header": [...],
        "data_rows": [...]
    }
    """
    headers = table["header"]
    rows = table["data_rows"]

    # Header
    markdown = "| " + " | ".join(str(h) for h in headers) + " |\n"

    # Separator
    markdown += "| " + " | ".join("---" for _ in headers) + " |\n"

    # Rows
    for row in rows:
        markdown += "| " + " | ".join(str(cell) for cell in row) + " |\n"

    return markdown

In [20]:
md_of_table = table_to_markdown(all_tables[0])

In [21]:
print(md_of_table)

| Plan | Price | Users |
| --- | --- | --- |
| Free | $0 | 1 |
| Pro | $20 | 5 |
| Team | $50 | 20 |
| Enterprise | $200 | 100 |



In [22]:
client = OpenAI()

def summarize_table(table_markdown):
    prompt = f"""
Summarize the following table concisely.
Preserve all important information, numbers, names, and relationships.
Do not add information that is not present in the table.

Table:
{table_markdown}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [23]:
print(summarize_table(md_of_table))

The table outlines four subscription plans with their respective prices and user limits:

- **Free**: $0 for 1 user
- **Pro**: $20 for 5 users
- **Team**: $50 for 20 users
- **Enterprise**: $200 for 100 users


# structure is important

Income Statement

| Year | Revenue | COGS | Gross Profit |
|------|---------|------|--------------|
| 2022 | $10M    | $6M  | $4M          |
| 2023 | $12M    | $7M  | $5M          |
| 2024 | $15M    | $8M  | $7M          |

In [12]:
md = """
Income Statement

| Year | Revenue | COGS | Gross Profit |
|------|---------|------|--------------|
| 2022 | $10M    | $6M  | $4M          |
| 2023 | $12M    | $7M  | $5M          |
| 2024 | $15M    | $8M  | $7M          |
    """

print(md)


Income Statement

| Year | Revenue | COGS | Gross Profit |
|------|---------|------|--------------|
| 2022 | $10M    | $6M  | $4M          |
| 2023 | $12M    | $7M  | $5M          |
| 2024 | $15M    | $8M  | $7M          |
    
